# Dissertation Statistics

Reproducible computations for the statistics cited in the methodology text.
Every number that appears in the thesis originates here.

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

DATA = Path("../data/processed")

## 1. Panel summary statistics

In [2]:
panel = pl.read_parquet(DATA / "panel.parquet")
print(f"Panel shape: {panel.shape[0]:,} rows × {panel.shape[1]} cols")
print(f"Symbols: {panel['symbol'].n_unique()}")
print(f"Date range: {panel['date'].min()} — {panel['date'].max()}")
print(f"Columns: {panel.columns}")

Panel shape: 189,006 rows × 47 cols
Symbols: 193
Date range: 2022-01-03 — 2026-03-19
Columns: ['symbol', 'date', 'rv_d', 'rv_w', 'rv_m', 'rq', 'rs_pos', 'rs_neg', 'rv_21d_forward', 'mom_5d', 'mom_22d', 'mom_63d', 'mom_126d', 'mom_252d', 'max_daily_ret', 'rsi_14', 'ma_cross_1_9', 'ma_cross_2_12', 'atm_iv', 'iv_skew', 'iv_term_slope', 'tlm30', 'pc_volume_ratio', 'pc_oi_ratio', 'rn_skewness', 'rn_kurtosis', 'vs_level', 'vs_change', 'option_turnover', 'vrp', 'vrp_ratio', 'vol_of_vol', 'volunc', 'days_to_fomc', 'is_fomc_week', 'days_to_earnings', 'is_earnings_week', 'days_to_cpi', 'is_cpi_week', 'vix', 'term_spread', 'credit_spread', 'epu', 'tbill_change', 'ads_index', 'vvix', 'hsi_overnight_vol']


## 2. Filtering steps — row counts

In [3]:
from pathlib import Path
import polars as pl

RAW = Path("../data/raw")
PROC = Path("../data/processed")

def count_parquet_rows(folder: Path) -> int:
    total = 0
    for f in sorted(folder.glob("*.parquet")):
        total += pl.scan_parquet(f).select(pl.len()).collect().item()
    return total

stages = {
    "1. Raw EOD": (RAW / "eod", None),
    "2. Joined": (PROC / "joined", None),
    "3. Filtered (liquidity + DTE + monthly)": (PROC / "options_clean", None),
    "4. IV + delta filter": (PROC / "options_iv", None),
    "5. Features (aggregated, daily)": (PROC / "features", None),
}

print(f"{'Stage':<45} {'Rows':>15} {'Files':>6}")
print("-" * 70)
prev = None
for name, (folder, _) in stages.items():
    files = list(folder.glob("*.parquet"))
    n = count_parquet_rows(folder)
    surv = f"({n/prev*100:.1f}%)" if prev else ""
    print(f"{name:<45} {n:>15,} {len(files):>6} {surv}")
    prev = n

# Panel (final)
n_panel = panel.shape[0]
print(f"{'6. Panel (final)':<45} {n_panel:>15,}      1")

Stage                                                    Rows  Files
----------------------------------------------------------------------


1. Raw EOD                                        313,612,047    197 


2. Joined                                         245,843,603    193 (78.4%)


3. Filtered (liquidity + DTE + monthly)            38,877,748    193 (15.8%)


4. IV + delta filter                               15,778,186    193 (40.6%)


5. Features (aggregated, daily)                       190,987    193 (1.2%)
6. Panel (final)                                      189,006      1


## 3. Target distribution (skewness, log transform)

In [4]:
target = panel["rv_21d_forward"].to_numpy()
log_target = np.log(target)

from scipy import stats as sp_stats

print("=== rv_21d_forward (raw) ===")
print(f"  Mean:     {target.mean():.4f}")
print(f"  Median:   {np.median(target):.4f}")
print(f"  Std:      {target.std():.4f}")
print(f"  Skewness: {sp_stats.skew(target):.2f}")
print(f"  Kurtosis: {sp_stats.kurtosis(target):.2f}")
print()
print("=== log(rv_21d_forward) ===")
print(f"  Mean:     {log_target.mean():.4f}")
print(f"  Median:   {np.median(log_target):.4f}")
print(f"  Std:      {log_target.std():.4f}")
print(f"  Skewness: {sp_stats.skew(log_target):.2f}")
print(f"  Kurtosis: {sp_stats.kurtosis(log_target):.2f}")

=== rv_21d_forward (raw) ===
  Mean:     0.1715
  Median:   0.0842
  Std:      0.2788
  Skewness: 5.64
  Kurtosis: 49.97

=== log(rv_21d_forward) ===
  Mean:     -2.3870
  Median:   -2.4747
  Std:      1.0540
  Skewness: 0.42
  Kurtosis: -0.06


## 4. vs_level / iv_skew correlation and partial correlation

In [5]:
# Pearson correlation
vs = panel["vs_level"].to_numpy()
skew = panel["iv_skew"].to_numpy()
tgt = panel["rv_21d_forward"].to_numpy()

corr_vs_skew = np.corrcoef(vs, skew)[0, 1]
print(f"Pearson corr(vs_level, iv_skew) = {corr_vs_skew:.4f}")

# Bivariate correlations with target
corr_vs_tgt = np.corrcoef(vs, tgt)[0, 1]
corr_skew_tgt = np.corrcoef(skew, tgt)[0, 1]
print(f"Pearson corr(vs_level, target)  = {corr_vs_tgt:.4f}")
print(f"Pearson corr(iv_skew, target)   = {corr_skew_tgt:.4f}")

Pearson corr(vs_level, iv_skew) = -0.8471
Pearson corr(vs_level, target)  = 0.0555
Pearson corr(iv_skew, target)   = 0.0461


In [6]:
# Partial correlation: corr(vs_level, target | iv_skew)
# Formula: (r_xy - r_xz * r_yz) / sqrt((1 - r_xz^2)(1 - r_yz^2))
# where x=vs_level, y=target, z=iv_skew

r_xy = corr_vs_tgt
r_xz = corr_vs_skew
r_yz = corr_skew_tgt

partial_corr = (r_xy - r_xz * r_yz) / np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
print(f"Partial corr(vs_level, target | iv_skew) = {partial_corr:.4f}")
print()
print("Values cited in the thesis:")
print(f"  corr(vs_level, iv_skew) = −0.85  →  actual: {corr_vs_skew:.2f}")
print(f"  partial corr = +0.18             →  actual: {partial_corr:.2f}")

Partial corr(vs_level, target | iv_skew) = 0.1781

Values cited in the thesis:
  corr(vs_level, iv_skew) = −0.85  →  actual: -0.85
  partial corr = +0.18             →  actual: 0.18


## 5. VRP sign distribution

In [7]:
vrp = panel["vrp"].to_numpy()

n_pos = (vrp > 0).sum()
n_neg = (vrp < 0).sum()
n_zero = (vrp == 0).sum()
n_total = len(vrp)

print(f"VRP > 0:  {n_pos:>7,} ({n_pos/n_total*100:.1f}%)")
print(f"VRP < 0:  {n_neg:>7,} ({n_neg/n_total*100:.1f}%)")
print(f"VRP = 0:  {n_zero:>7,} ({n_zero/n_total*100:.1f}%)")
print(f"Mean VRP: {vrp.mean():.4f}")
print(f"Median:   {np.median(vrp):.4f}")
print()
print(f"→ 'VRP systematically positive': {'YES' if n_pos/n_total > 0.5 else 'NO'} ({n_pos/n_total*100:.0f}% positive)")

VRP > 0:  110,519 (58.5%)
VRP < 0:   78,487 (41.5%)
VRP = 0:        0 (0.0%)
Mean VRP: 0.0039
Median:   0.0177

→ 'VRP systematically positive': YES (58% positive)


## 6. SHAP top 10 — days_to_earnings ranking

In [8]:
# SHAP values
shap_vals = np.load(DATA / "models" / "lgbm_shap_values.npy")

# Feature columns: everything except symbol, date, target
exclude = {"symbol", "date", "rv_21d_forward"}
feature_cols = [c for c in panel.columns if c not in exclude]

mean_abs_shap = np.abs(shap_vals).mean(axis=0)
ranking = sorted(zip(feature_cols, mean_abs_shap), key=lambda x: -x[1])

print(f"{'Rank':<6} {'Feature':<25} {'Mean |SHAP|':>12}")
print("-" * 45)
for i, (feat, val) in enumerate(ranking[:15], 1):
    marker = " ← " if feat == "days_to_earnings" else ""
    print(f"{i:<6} {feat:<25} {val:>12.4f}{marker}")

# Find exact rank of days_to_earnings
dte_rank = next(i for i, (f, _) in enumerate(ranking, 1) if f == "days_to_earnings")
print(f"\ndays_to_earnings rank: #{dte_rank} (thesis: #8)")

Rank   Feature                    Mean |SHAP|
---------------------------------------------
1      rv_m                            0.5073
2      rq                              0.1470
3      rs_neg                          0.0545
4      rs_pos                          0.0475
5      atm_iv                          0.0387
6      rv_w                            0.0260
7      vol_of_vol                      0.0215
8      days_to_earnings                0.0210 ← 
9      max_daily_ret                   0.0187
10     volunc                          0.0108
11     days_to_fomc                    0.0068
12     epu                             0.0055
13     vrp                             0.0047
14     days_to_cpi                     0.0036
15     option_turnover                 0.0031

days_to_earnings rank: #8 (thesis: #8)


## 7. Feature correlation matrix (top collinearities)

In [9]:
# Full correlation matrix, top |corr| > 0.7 pairs
feat_matrix = panel.select(feature_cols).to_numpy()
corr_matrix = np.corrcoef(feat_matrix, rowvar=False)

pairs = []
n_feat = len(feature_cols)
for i in range(n_feat):
    for j in range(i + 1, n_feat):
        r = corr_matrix[i, j]
        if abs(r) > 0.7:
            pairs.append((feature_cols[i], feature_cols[j], r))

pairs.sort(key=lambda x: -abs(x[2]))

print(f"Feature pairs |corr| > 0.7 ({len(pairs)} pairs):")
print(f"{'Feature 1':<25} {'Feature 2':<25} {'Corr':>8}")
print("-" * 60)
for f1, f2, r in pairs:
    print(f"{f1:<25} {f2:<25} {r:>8.3f}")

Feature pairs |corr| > 0.7 (10 pairs):
Feature 1                 Feature 2                     Corr
------------------------------------------------------------
mom_5d                    ma_cross_1_9                 0.934
ma_cross_1_9              ma_cross_2_12                0.911
rv_m                      max_daily_ret                0.882
mom_5d                    ma_cross_2_12                0.855
iv_skew                   vs_level                    -0.847
rv_m                      rs_neg                       0.844
vrp                       vrp_ratio                    0.828
rv_m                      rs_pos                       0.825
rs_neg                    max_daily_ret                0.755
rs_pos                    max_daily_ret                0.716


## 8. Summary — verification of cited numbers

## 9. Seed ensemble QLIKE dispersion and CV QLIKE

In [10]:
print("=" * 65)
print("NUMBERS CITED IN THESIS vs ACTUAL")
print("=" * 65)

checks = [
    ("Panel rows", "189,006", f"{panel.shape[0]:,}"),
    ("Panel columns", "47", f"{panel.shape[1]}"),
    ("Symbols", "193", f"{panel['symbol'].n_unique()}"),
    ("Target skewness (raw)", "5.64", f"{sp_stats.skew(target):.2f}"),
    ("Target skewness (log)", "0.44", f"{sp_stats.skew(log_target):.2f}"),
    ("corr(vs_level, iv_skew)", "−0.85", f"{corr_vs_skew:.2f}"),
    ("partial corr(vs_level, target | iv_skew)", "+0.18", f"{partial_corr:.2f}"),
    ("VRP systematically positive", ">50%", f"{n_pos/n_total*100:.0f}%"),
    ("days_to_earnings SHAP rank", "#8", f"#{dte_rank}"),
]

for label, expected, actual in checks:
    match = "✓" if expected.replace("−", "-").replace("+", "").replace(">", "").strip("%") in actual.replace("-", "-").strip("%") or expected == actual else "?"
    print(f"  {match} {label:<45} thesis: {expected:<10} actual: {actual}")

NUMBERS CITED IN THESIS vs ACTUAL
  ✓ Panel rows                                    thesis: 189,006    actual: 189,006
  ✓ Panel columns                                 thesis: 47         actual: 47
  ✓ Symbols                                       thesis: 193        actual: 193
  ✓ Target skewness (raw)                         thesis: 5.64       actual: 5.64
  ? Target skewness (log)                         thesis: 0.44       actual: 0.42
  ✓ corr(vs_level, iv_skew)                       thesis: −0.85      actual: -0.85
  ✓ partial corr(vs_level, target | iv_skew)      thesis: +0.18      actual: 0.18
  ? VRP systematically positive                   thesis: >50%       actual: 58%
  ✓ days_to_earnings SHAP rank                    thesis: #8         actual: #8


In [11]:
import sys
sys.path.insert(0, str(Path("..").resolve()))

import torch
from theta.modeling.neural_models import (
    LSTMModel, VolatilitySequenceDataset,
    _load_scaler_stats, _standardize_df,
    _predict_batched, SPLITS_DIR, MODELS_DIR,
)
from theta.modeling.preprocessing import get_feature_cols, LOG_TARGET_COL
from theta.modeling.lightgbm_model import qlike_score

test = pl.read_parquet(SPLITS_DIR / "test.parquet")
feature_cols = get_feature_cols(test)
scaler_stats = _load_scaler_stats()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- FNN seed QLIKE ---
# fnn_ensemble.pt was not saved — only fnn_best_params.json and fnn.parquet (ensemble avg) exist.
# Seed range 0.0272–0.0353 is from original training output (not reproducible from saved artifacts).
print("=== FNN seed QLIKE ===")
print("  fnn_ensemble.pt not saved — seed values from original training log:")
print("  Seed 0: 0.0353, Seed 1: 0.0289, Seed 2: 0.0272, Seed 3: 0.0310, Seed 4: 0.0298")
print("  Range: 0.0272 – 0.0353")
print("  NOTE: not reproducible from saved artifacts, but ensemble result (0.0285) is in fnn.parquet")

# --- LSTM seed QLIKE (reproducible from saved checkpoint) ---
lstm_ckpt = torch.load(MODELS_DIR / "lstm_ensemble.pt", map_location=device, weights_only=False)
test_scaled = _standardize_df(test, feature_cols, scaler_stats)
lstm_ds = VolatilitySequenceDataset(test_scaled, feature_cols, seq_len=lstm_ckpt.get("seq_len", 21))
y_lstm_log = lstm_ds.y.numpy()

print("\n=== LSTM seed QLIKE ===")
lstm_seed_qlikes = []
for i, state in enumerate(lstm_ckpt["seed_states"]):
    model = LSTMModel(
        len(feature_cols),
        hidden_size=lstm_ckpt["best_params"].get("hidden_size", 32),
        num_layers=lstm_ckpt["best_params"].get("num_layers", 1),
        dropout=lstm_ckpt["best_params"].get("dropout", 0.2),
    )
    model.load_state_dict(state)
    model.to(device).eval()
    preds = _predict_batched(model, lstm_ds, device)
    q = qlike_score(y_lstm_log, preds)
    lstm_seed_qlikes.append(q)
    print(f"  Seed {i}: {q:.4f}")
print(f"  Range: {min(lstm_seed_qlikes):.4f} – {max(lstm_seed_qlikes):.4f}")

# --- LightGBM CV QLIKE ---
print("\n=== LightGBM best CV QLIKE ===")
print("  0.0208 (50 trials, 5-fold purged CV, from Optuna run)")
print("  Note: Optuna study not persisted — value from original training output")

print("\n=== Values cited in the thesis ===")
print(f"  FNN seed range:  thesis: 0.0272–0.0353  (training log, not reproducible)")
print(f"  LSTM seed range: thesis: 0.0163–0.0184  actual: {min(lstm_seed_qlikes):.4f}–{max(lstm_seed_qlikes):.4f}")
print(f"  LGBM CV QLIKE:   thesis: 0.0208         (training log, not reproducible)")

=== FNN seed QLIKE ===
  fnn_ensemble.pt not saved — seed values from original training log:
  Seed 0: 0.0353, Seed 1: 0.0289, Seed 2: 0.0272, Seed 3: 0.0310, Seed 4: 0.0298
  Range: 0.0272 – 0.0353
  NOTE: not reproducible from saved artifacts, but ensemble result (0.0285) is in fnn.parquet



=== LSTM seed QLIKE ===


  Seed 0: 0.0175


  Seed 1: 0.0163


  Seed 2: 0.0169


  Seed 3: 0.0184


  Seed 4: 0.0178
  Range: 0.0163 – 0.0184

=== LightGBM best CV QLIKE ===
  0.0208 (50 trials, 5-fold purged CV, from Optuna run)
  Note: Optuna study not persisted — value from original training output

=== Values cited in the thesis ===
  FNN seed range:  thesis: 0.0272–0.0353  (training log, not reproducible)
  LSTM seed range: thesis: 0.0163–0.0184  actual: 0.0163–0.0184
  LGBM CV QLIKE:   thesis: 0.0208         (training log, not reproducible)


## 10. Quarterly QLIKE breakdown + matched subset test

In [12]:
import polars as pl
import numpy as np
from pathlib import Path

PREDS = Path("../data/processed/predictions")

lstm = pl.read_parquet(PREDS / "lstm.parquet")
lgbm = pl.read_parquet(PREDS / "lgbm.parquet")
baselines = pl.read_parquet(PREDS / "baselines.parquet")
loghar = baselines.filter(pl.col("model") == "LogHAR")

# --- Quarterly QLIKE ---
print("=== Quarterly QLIKE ===\n")
for name, df in [("LSTM", lstm), ("LightGBM", lgbm), ("LogHAR", loghar)]:
    df2 = df.with_columns(
        (pl.col("date").dt.year().cast(pl.Utf8) + "-Q"
         + ((pl.col("date").dt.month() - 1) // 3 + 1).cast(pl.Utf8)).alias("quarter")
    )
    quarters = df2.group_by("quarter").agg([
        pl.len().alias("n"),
        (pl.col("y_true") / pl.col("y_pred")
         - (pl.col("y_true") / pl.col("y_pred")).log() - 1).mean().alias("qlike")
    ]).sort("quarter")
    print(f"  {name}:")
    for row in quarters.iter_rows(named=True):
        print(f"    {row['quarter']}: QLIKE={row['qlike']:.4f} (n={row['n']})")
    print()

# LSTM vs LightGBM quarterly advantage
print("=== LSTM vs LightGBM quarterly advantage ===")
for q, lq, gq in [(("2025-Q2"), 0.0230, 0.0246), ("2025-Q3", 0.0159, 0.0225),
                    ("2025-Q4", 0.0156, 0.0198), ("2026-Q1", 0.0152, 0.0202)]:
    pct = (gq - lq) / gq * 100
    print(f"  {q}: {pct:.1f}%")

# --- Matched subset test ---
print("\n=== Matched subset test ===\n")
lstm_keys = lstm.select("symbol", "date")

for model_name in ["LogHAR", "HAR", "HARQ", "SHAR", "LevHAR"]:
    df_full = baselines.filter(pl.col("model") == model_name)
    df_matched = df_full.join(lstm_keys, on=["symbol", "date"], how="inner")
    y_f, yh_f = df_full["y_true"].to_numpy(), df_full["y_pred"].to_numpy()
    y_m, yh_m = df_matched["y_true"].to_numpy(), df_matched["y_pred"].to_numpy()
    q_full = float(np.mean(y_f / yh_f - np.log(y_f / yh_f) - 1))
    q_match = float(np.mean(y_m / yh_m - np.log(y_m / yh_m) - 1))
    diff = (q_match - q_full) / q_full * 100
    print(f"  {model_name:>10}: full={q_full:.4f}  matched={q_match:.4f}  diff={diff:+.1f}%")

# --- LogHAR n=36811 explanation ---
print("\n=== LogHAR row loss ===")
har_full = baselines.filter(pl.col("model") == "HAR")
loghar_keys = loghar.select("symbol", "date")
har_keys = har_full.select("symbol", "date")
missing = har_keys.join(loghar_keys, on=["symbol", "date"], how="anti")
print(f"  HAR - LogHAR = {len(missing)} missing rows")

test = pl.read_parquet("../data/processed/splits/test.parquet")
missing_data = test.join(missing, on=["symbol", "date"], how="inner")
rv_d_zeros = (missing_data["rv_d"] == 0).sum()
print(f"  Of which rv_d == 0: {rv_d_zeros} (log(0) undefined)")

# --- FNN vs LightGBM percentage ---
print("\n=== Percentage differences ===")
print(f"  FNN vs LightGBM: +{(0.028486 - 0.021492) / 0.021492 * 100:.1f}%")
print(f"  LSTM vs LogHAR:  -{(0.025880 - 0.016018) / 0.025880 * 100:.1f}%")
print(f"  LSTM vs LightGBM: -{(0.021492 - 0.016018) / 0.021492 * 100:.1f}%")

=== Quarterly QLIKE ===

  LSTM:
    2025-Q2: QLIKE=0.0230 (n=1923)
    2025-Q3: QLIKE=0.0159 (n=11745)
    2025-Q4: QLIKE=0.0156 (n=11522)
    2026-Q1: QLIKE=0.0152 (n=7894)

  LightGBM:
    2025-Q2: QLIKE=0.0246 (n=5753)
    2025-Q3: QLIKE=0.0225 (n=11755)
    2025-Q4: QLIKE=0.0198 (n=11522)
    2026-Q1: QLIKE=0.0202 (n=7894)

  LogHAR:
    2025-Q2: QLIKE=0.0281 (n=5725)
    2025-Q3: QLIKE=0.0305 (n=11723)
    2025-Q4: QLIKE=0.0226 (n=11485)
    2026-Q1: QLIKE=0.0223 (n=7878)

=== LSTM vs LightGBM quarterly advantage ===
  2025-Q2: 6.5%
  2025-Q3: 29.3%
  2025-Q4: 21.2%
  2026-Q1: 24.8%

=== Matched subset test ===

      LogHAR: full=0.0259  matched=0.0261  diff=+0.9%
         HAR: full=0.0265  matched=0.0268  diff=+0.8%
        HARQ: full=0.0263  matched=0.0265  diff=+0.9%
        SHAR: full=0.0264  matched=0.0267  diff=+0.9%
      LevHAR: full=0.0265  matched=0.0268  diff=+0.9%

=== LogHAR row loss ===
  HAR - LogHAR = 113 missing rows
  Of which rv_d == 0: 113 (log(0) undefined)
